# Electricity Theft & Anomaly Detection
## Phase 1 – Business & Problem Analysis

Final Year Data Science Capstone Project

## 1. Business Context

Electricity Distribution Companies (DISCOMs) face significant financial losses due to:

- Electricity theft
- Meter tampering
- Illegal connections
- Billing manipulation

These losses are categorized as **Non-Technical Losses (NTL)**.

NTL can account for 15%–30% of total distributed energy in some regions.

Traditional rule-based detection systems are inefficient and fail to detect sophisticated theft patterns.

Hence, an AI-driven system is required.

## 2. Problem Statement

Design a real-time, data-driven system that:

- Detects abnormal electricity consumption behavior
- Predicts probability of electricity theft
- Minimizes false inspection costs
- Provides explainable decisions
- Reduces Non-Technical Loss (NTL)

## 3. Business Objectives

1. Reduce NTL by at least 8–10%
2. Improve inspection efficiency
3. Prioritize high-risk consumers
4. Increase revenue recovery
5. Provide regulatory-compliant explanations

In [5]:
import pandas as pd

df = pd.read_csv("../data/processed/feature_engineered.csv")

df.head()


,consumer_id,timestamp,consumption_kwh,voltage,current,power_factor,theft_label,drift_flag,temperature,humidity,...,rolling_mean_6h,rolling_std_6h,rolling_mean_12h,rolling_std_12h,rolling_mean_24h,rolling_std_24h,lag_1h,lag_24h,hour,dayofweek
0,1,2024-01-01 00:00:00,1.408501,231.134680,6.600922,0.970170,0,0,25.120493,65.814074,...,1.594694,0.308976,1.661295,0.27105,1.451139,0.316325,1.408501,1.408501,0,0
1,1,2024-01-01 01:00:00,1.590901,226.709743,6.022453,0.949561,0,0,25.120493,64.443713,...,1.594694,0.308976,1.661295,0.27105,1.451139,0.316325,1.408501,1.408501,1,0
2,1,2024-01-01 02:00:00,1.539829,233.289587,6.373323,0.967279,0,0,25.120493,83.242515,...,1.594694,0.308976,1.661295,0.27105,1.451139,0.316325,1.590901,1.408501,2,0
3,1,2024-01-01 03:00:00,1.215253,233.507623,5.297672,0.879270,0,0,25.120493,62.890345,...,1.594694,0.308976,1.661295,0.27105,1.451139,0.316325,1.539829,1.408501,3,0
4,1,2024-01-01 04:00:00,2.131333,231.189122,8.212522,0.987504,0,0,25.120493,56.974080,...,1.594694,0.308976,1.661295,0.27105,1.451139,0.316325,1.215253,1.408501,4,0


In [6]:
print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns)


Dataset Shape: (287522, 25)

Columns:
Index(['consumer_id', 'timestamp', 'consumption_kwh', 'voltage', 'current',
       'power_factor', 'theft_label', 'drift_flag', 'temperature', 'humidity',
       'wind_speed', 'rainfall_mm', 'heat_index', 'is_extreme_weather',
       'weather_severity_score', 'rolling_mean_6h', 'rolling_std_6h',
       'rolling_mean_12h', 'rolling_std_12h', 'rolling_mean_24h',
       'rolling_std_24h', 'lag_1h', 'lag_24h', 'hour', 'dayofweek'],
      dtype='object')


In [7]:
theft_counts = df["theft_label"].value_counts()
print(theft_counts)

theft_percentage = (theft_counts[1] / len(df)) * 100
print(f"\nTheft Percentage: {theft_percentage:.2f}%")


theft_label
0    285655
1      1867
Name: count, dtype: int64

Theft Percentage: 0.65%


## 4. Data Imbalance

Electricity theft cases are rare events.

This leads to a **class imbalance problem**, where:

- Normal cases >> Theft cases

This requires:
- Class-weighted models
- SMOTE or resampling
- Cost-sensitive learning

In [8]:
df.describe()

,consumer_id,consumption_kwh,voltage,current,power_factor,theft_label,drift_flag,temperature,humidity,wind_speed,...,rolling_mean_6h,rolling_std_6h,rolling_mean_12h,rolling_std_12h,rolling_mean_24h,rolling_std_24h,lag_1h,lag_24h,hour,dayofweek
count,287522.000000,287522.000000,287522.000000,287522.000000,287522.000000,287522.000000,287522.000000,287522.000000,287522.000000,287522.000000,...,287522.000000,287522.000000,287522.000000,287522.000000,287522.000000,287522.000000,287522.000000,287522.000000,287522.000000,287522.000000
mean,100.493155,1.841306,229.997965,7.364703,0.920029,0.006493,0.300015,28.354849,64.647112,4.306405,...,1.841313,0.250161,1.841436,0.296382,1.838110,0.333964,1.841069,1.834097,11.489378,2.899802
std,57.724476,0.856449,3.004739,3.463219,0.040416,0.080320,0.458265,1.763494,11.571709,2.130740,...,0.819187,0.108002,0.802347,0.093465,0.786249,0.077197,0.856293,0.851737,6.922171,1.989112
min,1.000000,0.000063,215.531617,-1.643816,0.850001,0.000000,0.000000,25.120493,45.008801,0.500632,...,0.128943,0.016432,0.229165,0.066091,0.431587,0.126409,0.000063,0.000063,0.000000,0.000000
25%,51.000000,1.170229,227.974009,4.672883,0.884967,0.000000,0.000000,26.903707,54.916939,2.507436,...,1.180413,0.177826,1.182458,0.232170,1.163299,0.281014,1.170026,1.165768,5.000000,1.000000
50%,100.000000,1.816038,229.994500,7.261858,0.920158,0.000000,0.000000,28.456429,64.339002,4.311659,...,1.820027,0.231965,1.822073,0.281098,1.806816,0.315242,1.815836,1.810372,11.000000,3.000000
75%,150.000000,2.464002,232.020633,9.868228,0.955001,0.000000,1.000000,29.896208,74.262355,6.170011,...,2.451487,0.299452,2.447165,0.342469,2.455064,0.373066,2.463514,2.454503,17.000000,5.000000
max,200.000000,5.175656,243.753056,21.219519,0.990000,1.000000,1.000000,31.011348,84.994346,7.999942,...,4.549057,1.367837,4.251673,1.099946,3.989301,0.985823,5.175656,5.175656,23.000000,6.000000


## 5. Technical Challenges

1. Highly imbalanced dataset
2. Time-series nature of data
3. Concept drift over time
4. False positive cost
5. Need for explainability

---

## 6. Proposed Solution Strategy

We propose a hybrid system combining:

- Unsupervised Anomaly Detection
- Supervised Theft Classification
- Transformer-Level Analysis
- Explainable AI


## 7. Evaluation Metrics

### Technical Metrics
- Precision
- Recall
- F1 Score
- ROC-AUC

### Business Metrics
- Revenue recovered
- Inspection cost savings
- Reduction in NTL %


## 8. Hybrid Risk Score

Final Risk Score =

0.5 × Theft Probability  
+ 0.3 × Anomaly Score  
+ 0.2 × Transformer Suspicion Score


## 9. Conclusion

This project aims to build a scalable, explainable, and real-time electricity theft detection system.

The next phase involves:

- Exploratory Data Analysis (EDA)
- Feature Engineering
- Model Development
